a slight variation of SGD where the learning rate decays over training

In [1]:
from collections.abc import Callable, Iterable
from typing import Optional
import torch
import math
class SGD(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3):
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = {"lr": lr}
        super().__init__(params, defaults)
    def step(self, closure: Optional[Callable] = None):
        loss = None if closure is None else closure()
        for group in self.param_groups:
            lr = group["lr"] # Get the learning rate.
            for p in group["params"]:
                if p.grad is None:
                    continue
            state = self.state[p] # Get state associated with p.
            t = state.get("t", 0) # Get iteration number from the state, or initial value.
            grad = p.grad.data # Get the gradient of loss with respect to p.
            p.data -= lr / math.sqrt(t + 1) * grad # Update weight tensor in-place.
            state["t"] = t + 1 # Increment iteration number.
        return loss

minimal example of a training loop

In [ ]:
weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
opt = SGD([weights], lr=1)
for t in range(100):
    opt.zero_grad() # Reset the gradients for all learnable parameters.
    loss = (weights**2).mean() # Compute a scalar loss value.
    print(loss.cpu().item())
    loss.backward() # Run backward pass, which computes gradients.
    opt.step() # Run optimizer step.

Run the SGD example above with three other values for the learning rate: 1e1, 1e2, and 1e3, for just 10 training iterations. What happens with the loss for each of these learning rates? Does it decay faster, slower, or does it diverge (i.e., increase over the course of training)?

In [4]:
import pandas as pd

loss_show1 = []
loss_show2 = []
loss_show3 = []

weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
opt = SGD([weights], lr=1e1)
for t in range(10):
    opt.zero_grad() # Reset the gradients for all learnable parameters.
    loss = (weights**2).mean() # Compute a scalar loss value.
    # print(loss.cpu().item())

    loss_show1.append(loss.cpu().item())

    loss.backward() # Run backward pass, which computes gradients.
    opt.step() # Run optimizer step.

weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
opt = SGD([weights], lr=1e2)
for t in range(10):
    opt.zero_grad() # Reset the gradients for all learnable parameters.
    loss = (weights**2).mean() # Compute a scalar loss value.
    # print(loss.cpu().item())

    loss_show2.append(loss.cpu().item())

    loss.backward() # Run backward pass, which computes gradients.
    opt.step() # Run optimizer step.

weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
opt = SGD([weights], lr=1e3)
for t in range(10):
    opt.zero_grad() # Reset the gradients for all learnable parameters.
    loss = (weights**2).mean() # Compute a scalar loss value.
    # print(loss.cpu().item())

    loss_show3.append(loss.cpu().item())

    loss.backward() # Run backward pass, which computes gradients.
    opt.step() # Run optimizer step.


df = pd.DataFrame({
    'lr=1e1': loss_show1,
    'lr=1e2': loss_show2,
    'lr=1e3': loss_show3
})
print(df)

      lr=1e1        lr=1e2        lr=1e3
0  27.169418  2.500397e+01  3.596137e+01
1  17.388430  2.500397e+01  1.298205e+04
2  12.818007  4.290003e+00  2.242205e+06
3  10.028721  1.026694e-01  2.494214e+08
4   8.123264  2.133016e-16  2.020313e+10
5   6.735116  2.377379e-18  1.275049e+12
6   5.680178  8.005471e-20  6.545684e+13
7   4.853873  4.768914e-21  2.816233e+15
8   4.191702  4.091082e-22  1.038003e+17
9   3.651438  4.545647e-23  3.333145e+18
